In [ ]:
import sys
import os
from typing import NamedTuple
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt

from sklearn.cross_decomposition import CCA
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
import jax
import jax.numpy as jnp
import optax


from utils_fly import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../'))
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
dims = 2

# read_ids = list(range(0, 12+1))
# read_ids = [0,   # paper
# 			2,
# 			3,
# 			4,
# 			5,
# 			6,
# 			8,
# 			9]   # good
read_ids = [0]
plot_id = 0

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'brain')
(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep

def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

markers_grad = [pd.DataFrame(np.gradient(marker.values, axis=0), columns=marker.columns) for marker in markers]

In [ ]:
selection_bhl = ['walk', 'still', 'front-groom', 'back-groom', 'abdomen-move', 'background']

plot_idx = file_ids.index(plot_id)

selection_n = 5

# discrete_states = [ds[selection_bhl] for ds in discrete_states]

In [ ]:
plot_utils.plot_traces(traces[plot_idx].iloc[:, :selection_n],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'{filenames[plot_idx]} traces', ylimits=None)

# sklearn CCA

In [ ]:
X = traces[plot_idx].values
Y = states[plot_idx].values

In [ ]:
cca = CCA(n_components=dims, scale=True)
cca.fit(X, Y)
X_l_h, Y_l_h = cca.transform(X, Y)
X_rec_h, Y_rec_h = cca.inverse_transform(X_l_h, Y_l_h)

# def norm_temp(X_proj):
# 	X_proj = X_proj - np.mean(X_proj, axis=0)
# 	X_proj = X_proj / (np.linalg.norm(X_proj, axis=0) + 1e-6)
# 	return X_proj
# corr_a = np.array(norm_temp(X_l_h).T @ norm_temp(Y_l_h))

In [ ]:
X_l_h_df = pd.DataFrame(data=X_l_h, columns=[f'CV{i}' for i in range(dims)])

if dims > 1:
	plot_utils.plot_manifolds([filenames[plot_idx]],
							  [X_l_h_df],
							  [states[plot_idx]],
							  None, None,
							  title='art manifold', top_space=2.5,
							  save_path=None)

In [ ]:
plot_utils.plot_traces([X_l_h_df],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca neural modes', ylimits=None)

In [ ]:
# Y_l_h_df = pd.DataFrame(data=Y_l_h, columns=[f'CV{i}' for i in range(dims)])
#
# plot_utils.plot_traces([Y_l_h_df],
# 					   states[plot_idx][['still']],
# 					   label='quiescent',
# 					   title=f'[{filenames[plot_idx]}] cca behavioral modes', ylimits=None)

In [ ]:
X_rec_h_df = pd.DataFrame(data=X_rec_h, columns=traces[plot_idx].columns)

plot_utils.plot_traces([traces[plot_idx].iloc[:, :selection_n], X_rec_h_df.iloc[:, :selection_n]],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca neural reconstructions', ylimits=None)

In [ ]:
# Y_rec_h_df = pd.DataFrame(data=Y_rec_h, columns=states[plot_idx].columns)
#
# plot_utils.plot_traces([states[plot_idx], Y_rec_h_df],
# 					   states[plot_idx][['still']],
# 					   label='quiescent',
# 					   title=f'[{filenames[plot_idx]}] cca behavioral reconstructions', ylimits=None)

# soft CCA

In [ ]:
class Lambdas(NamedTuple):
	cca_lambdas: tuple[float, float]


def init_params(N, S, seeds, shape_U, shape_V):
	return {
		'prep': {
			'N': {
				'mean': N.mean(axis=0),
				'std': jnp.where((N_std := jnp.std(N, axis=0)) == 0.0, 1.0, N_std),
			},
			'S': {
				'mean': S.mean(axis=0),
				'std': jnp.where((S_std := jnp.std(S, axis=0)) == 0.0, 1.0, S_std),
			},
		},
		'model': {
			'U': 1E-2 * jax.random.normal(seeds[0], shape_U),
			'V': 1E-2 * jax.random.normal(seeds[1], shape_V)
		}
	}


#@jax.jit
def norm(X_proj):
	X_proj = X_proj - jnp.mean(X_proj, axis=0)
	X_proj = X_proj / (jnp.linalg.norm(X_proj, axis=0) + 1e-6)
	return X_proj

@jax.jit
def cca_loss(X_proj, Y_proj, ccadiag_lambda, ccaunit_lambda):
	corr = norm(X_proj).T @ norm(Y_proj)

	corr_diag = jnp.diag(jnp.diag(corr))
	corr_nondiag = corr - corr_diag

	return (ccadiag_lambda * jnp.mean(jnp.abs(corr_nondiag))
			+ ccaunit_lambda * jnp.mean(jnp.abs(corr_diag - jnp.eye(*corr_diag.shape))))

#@jax.jit
def loss_fn(params, N, S, lambdas):
	ccadiag_lambda, ccaunit_lambda = lambdas.cca_lambdas

	# manifold ======================================
	prep_n = jax.lax.stop_gradient(params['prep']['N'])
	prep_s = jax.lax.stop_gradient(params['prep']['S'])

	N_mean, N_std = prep_n['mean'], prep_n['std']
	S_mean, S_std = prep_s['mean'], prep_s['std']

	U = params['model']['U']
	V = params['model']['V']

	N_zs = (N - N_mean) / N_std
	S_zs = (S - S_mean) / S_std

	N_man = N_zs @ U
	S_man = S_zs @ V

	man_loss = cca_loss(N_man, S_man, ccadiag_lambda, ccaunit_lambda)
	# ===============================================

	return man_loss

loss_and_grads = jax.jit(jax.value_and_grad(loss_fn, has_aux=False), static_argnums=(3,))


def update(opt_update, opt_state, params, N, S, lambdas):
	loss, grads = loss_and_grads(params, N, S, lambdas)
	params_updates, opt_state = opt_update(grads, opt_state, params)
	params_new = optax.apply_updates(params, params_updates)
	return opt_state, params_new, loss

In [ ]:
def soft_cca_train(N, S, hyperparams):
	max_epochs = hyperparams['max_epochs']
	n_dimensions = hyperparams['dimensions']
	learning_rate = hyperparams['learning_rate']
	lambdas = Lambdas(cca_lambdas=(hyperparams['ccadiag_lambda'], hyperparams['ccaunit_lambda']))

	n_neurons = N.shape[1]
	n_behaviors_hl = S.shape[1]

	seeds = jax.random.PRNGKey(0), jax.random.PRNGKey(1)

	params = init_params(N, S, seeds, (n_neurons, n_dimensions), (n_behaviors_hl, n_dimensions))
	optimizer = optax.adam(learning_rate)
	optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
	opt_state = optimizer.init(params)

	losses_train = []
	for epoch in range(max_epochs):
		opt_state, params, loss_train = update(optimizer.update, opt_state, params, N, S, lambdas)
		losses_train.append(loss_train)

	return params, losses_train


def soft_cca_predict(params, N, S):

	def get_reconstruction_matrix(Y, X):
		# best reconstruction matrix that reconstruct X using Y
		# magic (actually just analytical least-squares solution to reconstruction)
		# return jnp.linalg.inv(Y.T @ Y) @ Y.T @ X
		# return jnp.linalg.solve(Y.T @ Y, Y.T @ X)
		return jnp.linalg.lstsq(Y, X, rcond=None)[0]
		# Y_hat = Y @ VB.T @ np.linalg.solve(VB @ VB.T + 1e-6*np.eye(VB.shape[0]), np.eye(VB.shape[0]))	# Given Y = X @ VB, what Y best explains X?


	prep_n = jax.lax.stop_gradient(params['prep']['N'])
	prep_s = jax.lax.stop_gradient(params['prep']['S'])

	N_mean, N_std = prep_n['mean'], prep_n['std']
	S_mean, S_std = prep_s['mean'], prep_s['std']

	U = params['model']['U']
	V = params['model']['V']

	N_zs = (N - N_mean) / N_std
	S_zs = (S - S_mean) / S_std

	N_man = N_zs @ U
	S_man = S_zs @ V

	N_zs_rec = N_man @ get_reconstruction_matrix(N_man, N_zs)
	S_zs_rec = S_man @ get_reconstruction_matrix(S_man, S_zs)

	N_rec = N_zs_rec * N_std + N_mean
	S_rec = S_zs_rec * S_std + S_mean

	return np.array(N_man), np.array(S_man), np.array(N_rec), np.array(S_rec)

In [ ]:
hyperparameters = {
	'max_epochs': 5000,
	'learning_rate': 1E-1,
	'ccadiag_lambda': 0.6,
	'ccaunit_lambda': 1.0 - 0.6,
	'dimensions': dims
}
params, losses_train = soft_cca_train(X, Y, hyperparameters)
X_l_s, Y_l_s, X_rec_s, Y_rec_s = soft_cca_predict(params, X, Y)

# corr_n = np.array(norming(X_l_s).T @ norming(Y_l_s) / norming(X_l_s).shape[0])

In [ ]:
losses = np.array(losses_train)

plot_utils.plot_loss(filenames[plot_idx],
					 losses,
					 title='soft cca', ylimits=0.2, plots_per_row=1)

In [ ]:
X_l_s_df = pd.DataFrame(data=X_l_s, columns=[f'CV{i}' for i in range(dims)])

if dims > 1:
	plot_utils.plot_manifolds([filenames[plot_idx]],
							  [X_l_s_df],
							  [states[plot_idx]],
							  None, None,
							  title='art manifold', top_space=2.5,
							  save_path=None)

In [ ]:
plot_utils.plot_traces([X_l_h_df, X_l_s_df],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca & soft cca neural modes', ylimits=None)

In [ ]:
# Y_l_s_df = pd.DataFrame(data=Y_l_s, columns=[f'CV{i}' for i in range(dims)])
#
# plot_utils.plot_traces([Y_l_h_df, Y_l_s_df],
# 					   states[plot_idx][['still']],
# 					   label='quiescent',
# 					   title=f'[{filenames[plot_idx]}] cca & soft cca behavioral modes', ylimits=None)

In [ ]:
X_rec_s_df = pd.DataFrame(data=X_rec_s, columns=traces[plot_idx].columns)

plot_utils.plot_traces([traces[plot_idx].iloc[:, :selection_n], X_rec_h_df.iloc[:, :selection_n], X_rec_s_df.iloc[:, :selection_n]],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca vs soft cca neural reconstructions using cinv', ylimits=None)

In [ ]:
# Y_rec_s_cinv_df = pd.DataFrame(data=Y_rec_s, columns=states[plot_idx].columns)
#
# plot_utils.plot_traces([states[plot_idx], Y_rec_h_df, Y_rec_s_cinv_df],
# 					   states[plot_idx][['still']],
# 					   label='quiescent',
# 					   title=f'[{filenames[plot_idx]}] cca vs soft cca behavioral reconstructions using cinv', ylimits=None)

In [ ]:
X_res_h_df = pd.DataFrame(data=X - X_rec_h, columns=traces[plot_idx].columns)
X_res_s_df = pd.DataFrame(data=X - X_rec_s, columns=traces[plot_idx].columns)

plot_utils.plot_traces([traces[plot_idx].iloc[:, :selection_n], X_res_h_df.iloc[:, :selection_n], X_res_s_df.iloc[:, :selection_n]],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca vs soft cca neural residuals', ylimits=None)

In [ ]:
plot_utils.plot_traces([X_res_h_df.iloc[:, :selection_n] - X_res_s_df.iloc[:, :selection_n]],
					   states[plot_idx][['still']],
					   label='quiescent',
					   title=f'[{filenames[plot_idx]}] cca vs soft cca neural residual differences', ylimits=None)